# Task 2 — Quantitative Analysis with TA-Lib & PyNance
Load historical stock price data, compute SMA, EMA, RSI, MACD, and visualize results.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

try:
    import talib
    TALIB_AVAILABLE = True
except ImportError:
    TALIB_AVAILABLE = False
    print('TA-Lib not installed — using pandas fallback for indicators')

plt.style.use('seaborn-v0_8-whitegrid')

# Stocks to analyze
TICKERS = ['AAPL', 'AMZN', 'GOOG', 'META', 'MSFT', 'NVDA', 'TSLA']

## 1. Load Stock Price Data

In [ ]:
import os

def load_stock(ticker):
    path = f'../data/raw/{ticker}_historical.csv'
    if not os.path.exists(path):
        # Fallback: download via yfinance
        import yfinance as yf
        df = yf.download(ticker, start='2020-01-01', end='2024-01-01', progress=False)
        df.to_csv(path)
    else:
        df = pd.read_csv(path, index_col=0, parse_dates=True)
    df.columns = [c.strip() for c in df.columns]
    df.dropna(inplace=True)
    return df

# Load one stock for demonstration
ticker = 'AAPL'
df = load_stock(ticker)
print(df.shape)
df.tail()

## 2. Compute Technical Indicators

In [ ]:
close = df['Close'].astype(float).values

if TALIB_AVAILABLE:
    df['SMA_20']  = talib.SMA(close, timeperiod=20)
    df['SMA_50']  = talib.SMA(close, timeperiod=50)
    df['EMA_20']  = talib.EMA(close, timeperiod=20)
    df['RSI_14']  = talib.RSI(close, timeperiod=14)
    df['MACD'], df['MACD_signal'], df['MACD_hist'] = talib.MACD(
        close, fastperiod=12, slowperiod=26, signalperiod=9
    )
else:
    # Pure-pandas fallback
    df['SMA_20'] = df['Close'].rolling(20).mean()
    df['SMA_50'] = df['Close'].rolling(50).mean()
    df['EMA_20'] = df['Close'].ewm(span=20, adjust=False).mean()

    delta = df['Close'].diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    rs    = gain / loss
    df['RSI_14'] = 100 - (100 / (1 + rs))

    ema12 = df['Close'].ewm(span=12, adjust=False).mean()
    ema26 = df['Close'].ewm(span=26, adjust=False).mean()
    df['MACD']        = ema12 - ema26
    df['MACD_signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
    df['MACD_hist']   = df['MACD'] - df['MACD_signal']

print('Indicators computed.')
df[['Close','SMA_20','SMA_50','EMA_20','RSI_14','MACD']].tail()

## 3. Visualize Price + Moving Averages

In [ ]:
plot_df = df.last('365D')  # last 1 year

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True,
                          gridspec_kw={'height_ratios': [3, 1, 1]})

# Panel 1: Price + MAs
axes[0].plot(plot_df.index, plot_df['Close'],  label='Close',   linewidth=1.2, color='black')
axes[0].plot(plot_df.index, plot_df['SMA_20'], label='SMA 20',  linewidth=1,   color='blue',   linestyle='--')
axes[0].plot(plot_df.index, plot_df['SMA_50'], label='SMA 50',  linewidth=1,   color='orange', linestyle='--')
axes[0].plot(plot_df.index, plot_df['EMA_20'], label='EMA 20',  linewidth=1,   color='green',  linestyle=':')
axes[0].set_title(f'{ticker} — Price & Moving Averages')
axes[0].set_ylabel('Price (USD)')
axes[0].legend()

# Panel 2: RSI
axes[1].plot(plot_df.index, plot_df['RSI_14'], color='purple', linewidth=1)
axes[1].axhline(70, color='red',   linestyle='--', linewidth=0.8, label='Overbought (70)')
axes[1].axhline(30, color='green', linestyle='--', linewidth=0.8, label='Oversold (30)')
axes[1].set_ylabel('RSI (14)')
axes[1].set_ylim(0, 100)
axes[1].legend(fontsize=8)

# Panel 3: MACD
axes[2].plot(plot_df.index, plot_df['MACD'],        label='MACD',   color='blue',   linewidth=1)
axes[2].plot(plot_df.index, plot_df['MACD_signal'], label='Signal', color='orange', linewidth=1)
axes[2].bar(plot_df.index,  plot_df['MACD_hist'],   label='Hist',   color='gray',   alpha=0.4, width=1)
axes[2].axhline(0, color='black', linewidth=0.5)
axes[2].set_ylabel('MACD')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig('../data/raw/fig7_technical_indicators.png', dpi=150)
plt.show()

## 4. PyNance — Additional Financial Metrics

In [ ]:
# Daily returns and rolling volatility (PyNance-style metrics via pandas)
df['daily_return']    = df['Adj Close'].pct_change() * 100
df['rolling_vol_20']  = df['daily_return'].rolling(20).std()  # 20-day volatility
df['rolling_vol_50']  = df['daily_return'].rolling(50).std()

# Sharpe-like ratio (annualized, risk-free=0 for simplicity)
mean_ret = df['daily_return'].mean()
std_ret  = df['daily_return'].std()
sharpe   = (mean_ret / std_ret) * np.sqrt(252)
print(f"Annualized Sharpe Ratio ({ticker}): {sharpe:.3f}")
print(f"Mean Daily Return: {mean_ret:.4f}%")
print(f"Daily Return Std Dev: {std_ret:.4f}%")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

axes[0].plot(df.index, df['daily_return'], linewidth=0.6, color='steelblue', alpha=0.8)
axes[0].axhline(0, color='black', linewidth=0.5)
axes[0].set_title(f'{ticker} — Daily Returns (%)')
axes[0].set_ylabel('Return (%)')

axes[1].plot(df.index, df['rolling_vol_20'], label='20-day Vol', color='red',    linewidth=1)
axes[1].plot(df.index, df['rolling_vol_50'], label='50-day Vol', color='orange', linewidth=1)
axes[1].set_title('Rolling Volatility')
axes[1].set_ylabel('Std Dev (%)')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/raw/fig8_returns_volatility.png', dpi=150)
plt.show()

## 5. Multi-Stock Summary

In [ ]:
summary = []
for t in TICKERS:
    try:
        d = load_stock(t)
        ret = d['Adj Close'].pct_change().dropna()
        summary.append({
            'Ticker': t,
            'Mean Daily Return (%)': round(ret.mean() * 100, 4),
            'Volatility (%)': round(ret.std() * 100, 4),
            'Sharpe': round((ret.mean() / ret.std()) * np.sqrt(252), 3),
            'Total Return (%)': round((d['Adj Close'].iloc[-1] / d['Adj Close'].iloc[0] - 1) * 100, 2)
        })
    except Exception as e:
        print(f'{t}: {e}')

pd.DataFrame(summary).set_index('Ticker')